## Hydrodynamic model of the Southern North Sea

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import mikeio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import modelskill as ms
import torch

# dataset helper
from helper import get_altimetry_obs, get_wl_point_obs
from helper import plot_shred_reconstruction, extract_water_level_measurements, plot_shred_reconstruction_multi

import sys
from pathlib import Path

# Add parent directory to path to access src module
sys.path.insert(0, str(Path.cwd().parent.parent))

# Importing modules from src
from src.datamanager import DataManager
from src.engine import SHREDEngine
from src.train import fit, evaluate
from src.mSHRED.mSHRED import mSHRED
from src.utils import plot_training_curves, evaluate_model

# get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load Data and Measurement

In [3]:
# folders
dfs_fldr = "datasets/output/Area.dfsu"
msh_fldr = "datasets/input/Mesh_8000_elems_HD_v2.mesh"
obs_fldr  = "datasets/observations/"
msh = mikeio.open(msh_fldr)
dfs = mikeio.open(dfs_fldr)
geom = dfs.geometry

ds = dfs.read(time = slice("2022-01-01", "2023-10-30"))

# sensor coverage
alllist = get_wl_point_obs()

# For sparse data simulation
hf = ["Dover", "ZeebruggeWielingendok", "NWA", "K141"]
# availabilities 97-50-98-50
alllist = [s for s in alllist if s.name in hf]

# Extract the values and convert to torch tensors

measurements = extract_water_level_measurements(alllist, ds_time=ds.time, obs_fldr=obs_fldr)
measurements = torch.as_tensor(measurements, dtype=torch.float32)
print("IN:\nMeasurements shape:", measurements.shape)

# Build a single mask for nodes that are valid in all output fields
surface = ds["Surface elevation"].values
u_velocity = ds["U velocity"].values
v_velocity = ds["V velocity"].values
valid_nodes = np.isfinite(surface).all(axis=0) & np.isfinite(u_velocity).all(axis=0) & np.isfinite(v_velocity).all(axis=0)

# outputs
H = torch.from_numpy(surface[:, valid_nodes].astype(np.float32, copy=False))
U = torch.from_numpy(u_velocity[:, valid_nodes].astype(np.float32, copy=False))
V = torch.from_numpy(v_velocity[:, valid_nodes].astype(np.float32, copy=False))

# print the shape of the arrays
print("OUT:\nH shape:", H.shape)
print("U shape:", U.shape)
print("V shape:", V.shape)

# free memory
del surface, u_velocity, v_velocity, valid_nodes, dfs, alllist

IN:
Measurements shape: torch.Size([32064, 4])
OUT:
H shape: torch.Size([32064, 8533])
U shape: torch.Size([32064, 8533])
V shape: torch.Size([32064, 8533])


In [4]:
ERRORS = []

# ROMDataManager
LAGS = 24

TRAIN_SIZE = 0.8        # percentage of data used for training 
VAL_SIZE = 0.1          # percentage of data used for validation
TEST_SIZE = 0.1         # percentage of data used for testing

COMPRESS_H = 12         # modes kept for H (surface elevation)
COMPRESS_U = 40         # modes kept for U (U velocity)
COMPRESS_V = 68         # modes kept for V (V velocity)

# Training
OPTIMIZER = "AdamW"     # optimizer to use
LEARNING_RATE = 5e-4    # learning rate for training
NUM_EPOCHS = 200        # maximum number of epochs for training
LR_SCHEDULER = "CosineAnnealingLR"      # learning rate scheduler to use
SCHEDULER_STEP_SIZE = NUM_EPOCHS        # step size for learning rate scheduler (if applicable)

BATCH_SIZE = 32         # batch size for training
PATIENCE = 40           # number of epochs with no improvement on validation loss before stopping training
VERBOSE_TRAIN = True

# Architecture
ENCODER_DEPTH = 3               # number of encoder layers
HIDDEN_SIZE = 32                # hidden size for encoder and decoder layers
N_HEADS = 4                     # number of attention heads (if using a transformer-based architecture)
DIM_FEEDFORWARD = 48            # dimension of feedforward network in transformer (if using a transformer-based architecture)
DECODER = "cnn"                 # type of decoder to use (mpl or cnn)   
DECODER_HIDDEN_SIZES = [64,96]     # hidden sizes for decoder layers 
DROPOUT = 0.1                   # dropout rate for encoder and decoder layers
MASKING_TYPE = "mask_data"      # type of masking to apply to inputs during training (mask_data, mask_data_random, etc.)

## (T.3.G): Masking x2 channels + MHSA learned

In [5]:
# Data manager
manager = DataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements)
manager.add_data(U, id="U velocity", compress=COMPRESS_U)
manager.add_data(V, id="V velocity", compress=COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare(verbose=True, masking_type=MASKING_TYPE)

# network input and output shapes
nn_input_shape = train_dataset.X.shape[-1]
nn_output_shape = train_dataset.Y.shape[-1]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)


Datasets: ['Surface elevation', 'U velocity', 'V velocity']

Number of nan values in sensor measurements after masking: 25037

Data shapes after preparation:
Train:	 torch.Size([25651, 24, 4]) -> torch.Size([25651, 120])
Val:	 torch.Size([3206, 24, 4]) -> torch.Size([3206, 120])
Test:	 torch.Size([3207, 24, 4]) -> torch.Size([3207, 120])

Input shape for the model: 4
Output shape for the model: 120


In [6]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_learned_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=True,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

In [7]:
# Load the trained model
mshred.load_state_dict(torch.load("models_saved/models_final/T3G.pth"))

<All keys matched successfully>

In [8]:
# mSHRED RECONSTRUCTION
engine = SHREDEngine(manager, mshred)
test_reconstruction = engine.sensor_to_reconstruction_dataset(test_dataset)

del test_dataset, engine, mshred, train_dataset, val_dataset

In [ ]:
plot_shred_reconstruction_multi(
    test_reconstruction, ds, geom,
    time_idxs=(-3601+480, -2401, -1201, -1),
    row_gap=1.3,
    cbar_gap=0.5,
    cbar_tick_pt=22,
    cbar_unit_pt=24,
    title_pad=0.2,
    title="HYDRO_reconstruction_grid",
)


In [9]:
from pathlib import Path
from src.utils import make_video

n_frames = min(30 * 24, test_reconstruction["Surface elevation"].shape[0])
output_dir = Path("video")
output_dir.mkdir(exist_ok=True)

for item, cmap, filename in [
    ("Surface elevation", "terrain", "surface_elevation"),
    ("U velocity", "twilight_shifted", "u_velocity"),
    ("V velocity", "twilight_shifted", "v_velocity"),
]:
    valid_columns = ~np.isnan(ds[item].values).any(axis=0)
    ground_truth = ds[item].values[-n_frames:]
    prediction = np.full_like(ground_truth, np.nan)
    prediction[:, valid_columns] = test_reconstruction[item][-n_frames:]
    error = np.abs(ground_truth - prediction)
    vmin = np.nanmin(np.concatenate([ground_truth, prediction]))
    vmax = np.nanmax(np.concatenate([ground_truth, prediction]))
    error_max = np.nanmax(error)

    def plot_hydro_snapshot(snapshot, cmap=cmap, limits=(vmin, vmax), is_error=False):
        ds_frame = mikeio.DataArray(data=snapshot, time=[ds.time[-1]], item=ds[item].item, geometry=geom)
        ds_frame.plot(ax=plt.gca(), cmap="Greys" if is_error else cmap, vmin=0 if is_error else limits[0], vmax=error_max if is_error else limits[1], add_colorbar=False)

    make_video([ground_truth], lambda snapshot: plot_hydro_snapshot(snapshot), titles=["Ground truth"], fontsize=18, title_weight="bold", figsize=(7, 5), save=True, name=output_dir / f"hydro_{filename}_ground_truth_last_month.gif", fps=10, transparent=True)
    print("video 1/3")
    make_video([prediction], lambda snapshot: plot_hydro_snapshot(snapshot), titles=["Masked SHRED"], fontsize=18, title_weight="bold", figsize=(7, 5), save=True, name=output_dir / f"hydro_{filename}_prediction_last_month.gif", fps=10, transparent=True)
    print("video 1/3")
    make_video([error], lambda snapshot: plot_hydro_snapshot(snapshot, is_error=True), titles=["Absolute error"], fontsize=18, title_weight="bold", figsize=(7, 5), save=True, name=output_dir / f"hydro_{filename}_error_last_month.gif", fps=10, transparent=True)
    print("video 1/3")
    print("field finished")
    
    del valid_columns, ground_truth, prediction, error, error_max


video 1/3
video 1/3
video 1/3
field finished
video 1/3
video 1/3
video 1/3
field finished
video 1/3
video 1/3
video 1/3
field finished
